# import

In [1]:
import sys
sys.path.append('..')
import cv2 as cv
import numpy as np
import pandas as pd
from nmot import NMOT
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
from pathlib import Path
from tqdm import tqdm

In [2]:
def track(
    input_path: str,
    output_path: str = "tracked_output.mp4",
    csv_path: str = "tracks.csv",
    imshow=True,
    roi=None,
    dist2Threshold=50,
    knn_history = 100,
    pred_head = 'kalman',
    warmup_frames = 50,
    kasdin_hurst=1.2
):
    if pred_head=='zero': pred_head=None
    
    cap = cv.VideoCapture(input_path)

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {input_path}")

    fps = cap.get(cv.CAP_PROP_FPS)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    if output_path is not None:
        fourcc = cv.VideoWriter_fourcc(*"mp4v")
        writer = cv.VideoWriter(output_path, fourcc, fps, (width, height))

    tracker = NMOT(
        warmup_frames=warmup_frames,
        dist2Threshold=dist2Threshold,
        knn_history=knn_history,
        min_area=8,
        max_area=80,
        max_match_dist=25,
        max_missed=12,
        pred_head=pred_head,
        roi=roi,
        kasdin_hurst=kasdin_hurst
    )

    while True:
        ok, frame = cap.read()

        if not ok:
            break

        vis, mask, active_tracks = tracker.update(frame)
        if output_path is not None: 
            writer.write(vis)

        if imshow:
            # cv2.imshow('frame', frame)
            fig, ax = plt.subplots(nrows=2)
            fig.set_size_inches((20, 10))
            
            ax[0].imshow(vis)
            ax[1].imshow(mask)
            
            plt.show()
            
            clear_output(wait=True)
            if cv.waitKey(1) & 0xFF == 27:
                cv.destroyAllWindows()
                if output_path:
                    writer.release()
                break

    cap.release()
    if output_path is not None: 
        writer.release()
    cv.destroyAllWindows()
    if csv_path:
        df = tracker.save_tracks(csv_path)

    # return df

# run

In [3]:
roi_dict = {'S1240004начало.MP4': (210, 70, 470, 340),
            'S1240005.MP4': (210, 60, 490, 350),
            'S1240006.MP4':(210, 60, 490, 360),
            'S1240007.MP4':(210, 60, 490, 360),
            'S1240008.MP4':(210, 60, 490, 360),
            'S1240009.MP4':(210, 60, 490, 360),
            'S1240010.MP4':(210, 60, 490, 360),
            'S1240012.MP4':(210, 60, 490, 360),
            'S1240013повороткреста.MP4':(210, 60, 490, 360),
            'S1240014приманка.MP4':(210, 60, 490, 360),
            'S1240015.MP4':(210, 60, 490, 360),
            'S1240016.MP4':(210, 60, 490, 360),
            'S1240017.MP4':(210, 60, 490, 360),
            'S1240018.MP4':(210, 60, 490, 360),
            'S1240019.MP4':(210, 60, 490, 360),
            'S1240020.MP4':(210, 60, 490, 360),
            'S1240021.MP4':(210, 60, 490, 360),
            'S1240022.MP4':(210, 60, 490, 360),
            'S1240023.MP4':(210, 60, 490, 360),
            'S1240024.MP4':(210, 60, 490, 360),
            'S1240025.MP4':(210, 60, 490, 360)}

In [4]:
frufa_dir  = Path(r'E:\asp\Ants\F.rufa')
frufa_save_dir  = Path(r'E:\asp\Ants\F.rufa\knn_kalman')
pbar = tqdm(frufa_dir.glob("*.MP4"), total=len(list(frufa_dir.glob("*.MP4"))))
for file_path in pbar:
    csv_path = frufa_save_dir.joinpath(file_path.with_suffix('.csv').name)
    output_path = frufa_save_dir.joinpath(file_path.with_stem(file_path.stem+'_tracks').with_suffix('.mp4').name)
    
    pbar.set_description(f"Processing {file_path.name}")

    track(input_path=file_path,
          output_path=output_path,
          csv_path=csv_path,
          dist2Threshold=200,
          knn_history=200,
          imshow=False,
          pred_head='kalman',
          warmup_frames=30,
          roi=roi_dict[file_path.name]
          )
    

Processing S1240025.MP4: 100%|██████████| 21/21 [1:48:04<00:00, 308.77s/it]          


In [5]:
pyeensis_dir  = Path(r'E:\asp\Ants\P.yeensis')
pyeensis_save_dir  = Path(r'E:\asp\Ants\P.yeensis\knn_kalman')
pbar = tqdm(pyeensis_dir.glob("*.MP4"), total=len(list(pyeensis_dir.glob("*.MP4"))))
for file_path in pbar:
    csv_path = pyeensis_save_dir.joinpath(file_path.with_suffix('.csv').name)
    output_path = pyeensis_save_dir.joinpath(file_path.with_stem(file_path.stem+'_tracks').with_suffix('.mp4').name)
    
    pbar.set_description(f"Processing {file_path.name}")

    track(input_path=file_path,
          output_path=output_path,
          csv_path=csv_path,
          dist2Threshold=200,
          knn_history=200,
          imshow=False,
          pred_head='kalman',
          warmup_frames=30,
          roi=None
          )
    

Processing S2190017.MP4: 100%|██████████| 56/56 [5:17:13<00:00, 339.89s/it]    


In [6]:
# pbar = tqdm(dir_path.glob("*.MP4"))
# for file_path in pbar:
#     csv_path = save_dir.joinpath(file_path.with_suffix('.csv').name)
#     output_path = save_dir.joinpath(file_path.with_stem(file_path.stem+'_tracks').with_suffix('.mp4').name)
    
#     pbar.set_description(f"Processing {file_path.name}")
#     df = process_video(input_path=file_path,
#                        output_path=output_path,
#                        csv_path=csv_path,
#                        roi=None)
#     pbar.update(1)

In [7]:
# pbar = tqdm(dir_path.glob("*.MP4"))
# for file_path in pbar:
#     csv_path = save_dir.joinpath(file_path.with_suffix('.csv').name)
#     output_path = save_dir.joinpath(file_path.with_stem(file_path.stem+'_tracks').with_suffix('.mp4').name)
    
#     pbar.set_description(f"Processing {file_path.name}")
#     df = process_video(input_path=file_path,
#                        output_path=output_path,
#                        csv_path=csv_path,
#                        roi=None)
#     pbar.update(1)

In [8]:
# import os
# from pathlib import Path
# from tqdm import tqdm
# dir_path = Path(r'/home/akhiyarov/asp/NMOT/data/synth_gen_gt')
# save_dir = Path(r'/home/akhiyarov/asp/NMOT/data/synth_gen_tracked')

In [9]:
# roi_dict = {'S1240004начало.MP4': (210, 70, 470, 340),
#             'S1240005.MP4': (210, 60, 490, 350),
#             'S1240006.MP4':(210, 60, 490, 360),
#             'S1240007.MP4':(210, 60, 490, 360),
#             'S1240008.MP4':(210, 60, 490, 360),
#             'S1240009.MP4':(210, 60, 490, 360),
#             'S1240010.MP4':(210, 60, 490, 360),
#             'S1240012.MP4':(210, 60, 490, 360),
#             'S1240013повороткреста.MP4':(210, 60, 490, 360),
#             'S1240014приманка.MP4':(210, 60, 490, 360),
#             'S1240015.MP4':(210, 60, 490, 360),
#             'S1240016.MP4':(210, 60, 490, 360),
#             'S1240017.MP4':(210, 60, 490, 360),
#             'S1240018.MP4':(210, 60, 490, 360),
#             'S1240019.MP4':(210, 60, 490, 360),
#             'S1240020.MP4':(210, 60, 490, 360),
#             'S1240021.MP4':(210, 60, 490, 360),
#             'S1240022.MP4':(210, 60, 490, 360),
#             'S1240023.MP4':(210, 60, 490, 360),
#             'S1240024.MP4':(210, 60, 490, 360),
#             'S1240025.MP4':(210, 60, 490, 360)}

In [10]:
# import pickle
# with open('frufa_roi.pkl', 'wb') as f:
#     pickle.dump(roi_dict, f)